# This notebook serves as a template for cleaning the original text data and saving the train/test splits to Parquet files rather than csv

In [1]:
# !pip install nltk

# !pip install pyspellchecker

# !pip install textblob

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [3]:
train.head()

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [4]:
test.head()

,essay_id,full_text
0,000d118,Many people have car where they live. The thin...
1,000fe60,I am a scientist at NASA that is discussing th...
2,001ab80,People always wish they had the same technolog...


In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob

# # Set the download location to Kaggle's working directory
# download_dir = '/kaggle/working/nltk_data'

# # Add this download directory to nltk's data path
# if download_dir not in nltk.data.path:
#     nltk.data.path.append(download_dir)

# Download necessary datasets from NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')
nltk.download('universal_tagset')
nltk.download('maxent_ne_chunker')
nltk.download('words')

# Now check if the directory is correctly set and files are present
# import os
# print(os.listdir(download_dir))  # This should show the downloaded files


[nltk_data] Downloading package punkt to /home/laptop/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/laptop/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/laptop/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/laptop/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /home/laptop/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/laptop/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /home/laptop/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nl

True

In [6]:
# import zipfile
# import os

# # Path to the zip file
# zip_path = '/kaggle/working/nltk_data/corpora/wordnet.zip'

# # Target extraction directory
# extract_dir = '/kaggle/working/nltk_data/corpora/wordnet'

# # Create the target directory if it doesn't already exist
# if not os.path.exists(extract_dir):
#     os.makedirs(extract_dir)

# # Extract the zip file
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

# # Verify the files have been extracted
# print(os.listdir(extract_dir))  # Should show the contents of the wordnet corpus


In [7]:
# import os
# import shutil

# # Set the source and target directories
# source_dir = '/kaggle/working/nltk_data/corpora/wordnet/wordnet'
# target_dir = '/kaggle/working/nltk_data/corpora/wordnet'

# # Move each file and subdirectory from the source to the target directory
# for filename in os.listdir(source_dir):
#     source_file = os.path.join(source_dir, filename)
#     target_file = os.path.join(target_dir, filename)
#     if os.path.isdir(source_file):
#         if os.path.exists(target_file):
#             shutil.rmtree(target_file)  # Remove if target directory already exists
#         shutil.move(source_file, target_dir)
#     else:
#         if os.path.exists(target_file):
#             os.remove(target_file)  # Remove if target file already exists
#         shutil.move(source_file, target_dir)

# # Clean up the now empty source directory
# os.rmdir(source_dir)

# # Verify the structure
# print(os.listdir(target_dir))  # Should list 'lexnames' among other files


In [8]:
# import os
# from tqdm import tqdm

# # List of file paths to unzip
# embeddings = ['data/archive.zip']

# def unzip_embeddings(file_paths):
#     """
#     Unzip a list of files.

#     Args:
#         file_paths (list): List of file paths to unzip.

#     Returns:
#         None
#     """
#     # Initialize tqdm with the total number of files to unzip
#     with tqdm(total=len(file_paths)) as pbar:
#         for emb in file_paths:
#             # Use the -o flag to automatically replace files
#             if os.system(f'unzip -o {emb} -d data/') == 0:
#                 print(f"Inflating {emb} successful.")
#             else:
#                 print(f"Inflating {emb} failed.")
#             pbar.update(1)  # Update the progress bar

# # Call the function to unzip files
# unzip_embeddings(embeddings)

In [9]:
# from tqdm import tqdm
# import numpy as np

# def embedding_checks(df, glove_path, paragram_path, wiki_news_path, col_name='clean_text'):
#     """
#     Load embeddings, build vocabulary from the DataFrame, and check the vocabulary coverage in the embeddings.
    
#     :param df: DataFrame containing the text data.
#     :param glove_path: Path to the GloVe embedding file.
#     :param paragram_path: Path to the Paragram embedding file.
#     :param wiki_news_path: Path to the Wiki News embedding file.
#     :param col_name: Column name of the DataFrame to analyze.
#     :return: Tuple of DataFrame, OOV words for GloVe, Paragram, and Wiki News embeddings.
#     """

#     def load_embed(file):
#         """
#         Load the embeddings from a file.
#         """
#         print(f"Loading embeddings from {file}")
        
#         def get_coefs(word, *arr): 
#             return word, np.asarray(arr, dtype='float32')
        
#         if file == wiki_news_path:
#             embeddings_index = dict(get_coefs(*o.split(" ")) for o in tqdm(open(file), "Reading Embedding File") if len(o)>100)
#         else:
#             embeddings_index = dict(get_coefs(*o.split(" ")) for o in tqdm(open(file, encoding='latin'), "Reading Embedding File"))
        
#         print(f"Loaded embeddings from {file}")
#         return embeddings_index



#     # Load embeddings
#     print("Loading all embeddings.")
#     embed_glove = load_embed(glove_path)
#     embed_paragram = load_embed(paragram_path)
#     embed_fasttext = load_embed(wiki_news_path)
#     print("All embeddings loaded.")
    
#     def build_vocab(texts):
#         """
#         Build a vocabulary from a given list of texts.
#         """
#         print("Building vocabulary.")
#         sentences = texts.apply(lambda x: x.split()).values
#         vocab = {}
#         for sentence in tqdm(sentences, desc="Populating Vocabulary"):
#             for word in sentence:
#                 vocab[word] = vocab.get(word, 0) + 1
#         print("Vocabulary built.")
#         return vocab

#     def check_coverage(vocab, embeddings_index):
#         """
#         Check which words in the vocabulary are covered by the embeddings.
#         """
#         print("Checking coverage.")
#         known_words = {}
#         unknown_words = {}
#         for word in tqdm(vocab.keys(), desc="Checking Words"):
#             if word in embeddings_index:
#                 known_words[word] = vocab[word]
#             else:
#                 unknown_words[word] = vocab[word]
#         print("Coverage checked.")
#         return sorted(unknown_words.items(), key=lambda x: x[1], reverse=True)

#     # Build and check vocab for the provided DataFrame column
#     print("Processing dataset.")
#     vocab = build_vocab(df[col_name])
    
#     oov_glove = check_coverage(vocab, embed_glove)
#     oov_paragram = check_coverage(vocab, embed_paragram)
#     oov_fasttext = check_coverage(vocab, embed_fasttext)
  
#     print("Processed dataset.")
    
#     return df, oov_glove, oov_paragram, oov_fasttext


In [10]:
from multiprocessing import Pool
from tqdm import tqdm
import numpy as np
import pandas as pd

def load_embed(file, wiki_news_path):
    """
    Load the embeddings from a file.
    """
    print(f"Loading embeddings from {file}")
    
    def get_coefs(word, *arr): 
        return word, np.asarray(arr, dtype='float32')
    
    if file == wiki_news_path:
        embeddings_index = dict(get_coefs(*o.split(" ")) for o in tqdm(open(file), "Reading Embedding File") if len(o)>100)
    else:
        embeddings_index = dict(get_coefs(*o.split(" ")) for o in tqdm(open(file, encoding='latin'), "Reading Embedding File"))
    
    print(f"Loaded embeddings from {file}")
    return embeddings_index

def parallel_load_embeddings(paths):
    """
    Load multiple embeddings in parallel using multiprocessing.
    
    :param paths: List of paths to the embedding files.
    :return: Dictionary of embeddings.
    """
    with Pool(processes=len(paths)) as pool:
        embeddings = pool.starmap(load_embed, [(path, paths[-1]) for path in paths])
    return dict(zip(["glove", "paragram", "fasttext"], embeddings))

def embedding_checks(df, glove_path, paragram_path, wiki_news_path, col_name='clean_text'):
    """
    Load embeddings, build vocabulary from the DataFrame, and check the vocabulary coverage in the embeddings.
    
    :param df: DataFrame containing the text data.
    :param glove_path: Path to the GloVe embedding file.
    :param paragram_path: Path to the Paragram embedding file.
    :param wiki_news_path: Path to the Wiki News embedding file.
    :param col_name: Column name of the DataFrame to analyze.
    :return: Tuple of DataFrame, OOV words for GloVe, Paragram, and Wiki News embeddings.
    """
    paths = [glove_path, paragram_path, wiki_news_path]
    embeddings = parallel_load_embeddings(paths)

    embed_glove, embed_paragram, embed_fasttext = embeddings.values()
    
    def build_vocab(texts):
        """
        Build a vocabulary from a given list of texts.
        """
        print("Building vocabulary.")
        sentences = texts.apply(lambda x: x.split()).values
        vocab = {}
        for sentence in tqdm(sentences, desc="Populating Vocabulary"):
            for word in sentence:
                vocab[word] = vocab.get(word, 0) + 1
        print("Vocabulary built.")
        return vocab

    def check_coverage(vocab, embeddings_index):
        """
        Check which words in the vocabulary are covered by the embeddings.
        """
        print("Checking coverage.")
        known_words = {}
        unknown_words = {}
        for word in tqdm(vocab.keys(), desc="Checking Words"):
            if word in embeddings_index:
                known_words[word] = vocab[word]
            else:
                unknown_words[word] = vocab[word]
        print("Coverage checked.")
        return sorted(unknown_words.items(), key=lambda x: x[1], reverse=True)

    print("Processing dataset.")
    vocab = build_vocab(df[col_name])
    
    oov_glove = check_coverage(vocab, embed_glove)
    oov_paragram = check_coverage(vocab, embed_paragram)
    oov_fasttext = check_coverage(vocab, embed_fasttext)
  
    print("Processed dataset.")
    
    return df, oov_glove, oov_paragram, oov_fasttext


In [11]:
# Preprocessing
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re

def clean_text(df, col_name = 'full_text'):
    """
    Preprocesses text for both training and testing datasets. 
    Includes loading embeddings, building vocabularies, cleaning text among other things.
    
    :param summaries_train: DataFrame with the training data
    :param summaries_test: DataFrame with the testing data
    :param glove_path: path to the GloVe embedding
    :param paragram_path: path to the Paragram embedding
    :param wiki_news_path: path to the Wiki News embedding
    
    :return: Preprocessed DataFrame and list of out-of-vocab words
    """
    print("Starting text cleaning process. \n")


    
    # Lowercase all texts

    df['lowered'] = df[col_name].apply(lambda x: x.lower())

    
    
    def clean_spacing(text):
        
        # Remove spaces before punctuation
        text = re.sub(r'\s+([,.!?])', r'\1', text)

        # Ensure there is one space after punctuation
        text = re.sub(r'([,.!?])([^\s])', r'\1 \2', text)

        return text
    

    df['clean_text'] = df['lowered'].apply(lambda x: clean_spacing(x))


        
    punct = "/-'?!.,#$%\'()*+-/:;<=>@[\\]^_`{|}~" + '""“”’' + '∞θ÷α•à−β∅³π‘₹´°£€\×™√²—–&'
    
    punct_mapping = {"‘": "'", "₹": "e", "´": "'", "°": "", "€": "e", "™": "tm", "√": " sqrt ", "×": "x", "²": "2", "—": "-", "–": "-", "’": "'", "_": "-",
                     "`": "'", '“': '"', '”': '"', '“': '"', "£": "e", '∞': 'infinity', 'θ': 'theta', '÷': '/', 'α': 'alpha', '•': '.', 'à': 'a', '−': '-', 
                     'β': 'beta', '∅': '', '³': '3', 'π': 'pi', }

    def clean_special_chars(text, punct, mapping):
        for p in mapping:
            text = text.replace(p, mapping[p])
        for p in punct:
            text = text.replace(p, f' {p} ')
        specials = {'\u200b': ' ', '…': ' ... ', '\ufeff': '', 'करना': '', 'है': ''}  
        for s in specials:
            text = text.replace(s, specials[s])
        return text
    

    df['clean_text'] = df['clean_text'].apply(lambda x: clean_special_chars(x, punct, punct_mapping))

    df['clean_text'] = df['lowered'].apply(lambda x: clean_spacing(x))


    cont_map = {
        "ain't": "am not","aren't": "are not","can't": "cannot","can't've": "cannot have","'cause": "because",  "could've": "could have",
        "couldn't": "could not","couldn't've": "could not have","didn't": "did not","doesn't": "does not","don't": "do not","hadn't": "had not",
        "hadn't've": "had not have","hasn't": "has not",
        "haven't": "have not","he'd": "he would","he'd've": "he would have","he'll": "he will","he'll've": "he will have","he's": "he is",
        "how'd": "how did","how'd'y": "how do you","how'll": "how will","how's": "how is","I'd": "I would","I'd've": "I would have","I'll": "I will",
        "I'll've": "I will have","I'm": "I am","I've": "I have",
        "isn't": "is not","it'd": "it had","it'd've": "it would have","it'll": "it will", "it'll've": "it will have","it's": "it is","let's": "let us",
        "ma'am": "madam","mayn't": "may not",
        "might've": "might have","mightn't": "might not","mightn't've": "might not have","must've": "must have","mustn't": "must not",
        "mustn't've": "must not have","needn't": "need not","needn't've": "need not have","o'clock": "of the clock","oughtn't": "ought not",
        "oughtn't've": "ought not have","shan't": "shall not","sha'n't": "shall not",
        "shan't've": "shall not have","she'd": "she would","she'd've": "she would have","she'll": "she will","she'll've": "she will have","she's": "she is",
        "should've": "should have","shouldn't": "should not","shouldn't've": "should not have","so've": "so have","so's": "so is","that'd": "that would",
        "that'd've": "that would have","that's": "that is","there'd": "there had","there'd've": "there would have","there's": "there is",
        "they'd": "they would","they'd've": "they would have","they'll": "they will","they'll've": "they will have","they're": "they are",
        "they've": "they have","to've": "to have","wasn't": "was not","we'd": "we had",
        "we'd've": "we would have","we'll": "we will","we'll've": "we will have","we're": "we are","we've": "we have",
        "weren't": "were not","what'll": "what will","what'll've": "what will have",
        "what're": "what are","what's": "what is","what've": "what have","when's": "when is","when've": "when have",
        "where'd": "where did","where's": "where is","where've": "where have","who'll": "who will","who'll've": "who will have","who's": "who is",
        "who've": "who have","why's": "why is",
        "why've": "why have","will've": "will have","won't": "will not","won't've": "will not have","would've": "would have","wouldn't": "would not",
        "wouldn't've": "would not have","y'all": "you all","y'alls": "you alls","y'all'd": "you all would",
        "y'all'd've": "you all would have","y'all're": "you all are","y'all've": "you all have","you'd": "you had","you'd've": "you would have",
        "you'll": "you you will","you'll've": "you you will have","you're": "you are",  "you've": "you have"}

    c_re = re.compile('(%s)' % '|'.join(cont_map.keys()))

    def expandContractions(text, c_re=c_re):
        def replace(match):
            return cont_map[match.group(0)]
        return c_re.sub(replace, text)

    import inflect

    p = inflect.engine()
    
    def removeHTML(x):
        html=re.compile(r'<.*?>')
        return html.sub(r'',x)
    def dataPreprocessing(x):
        x = x.lower()
        x = removeHTML(x)
        x = re.sub("@\w+", '',x)
        x = re.sub(r'\d+', lambda match: p.number_to_words(match.group()), x)

        x = re.sub("'\d+", '',x)
        x = re.sub("\d+", '',x)
        x = re.sub("http\w+", '',x)
        x = re.sub(r"\s+", " ", x)
        x = expandContractions(x)
        x = re.sub(r"\.+", ".", x)
        x = re.sub(r"\,+", ",", x)
        x = x.strip()
        return x
    
    df['clean_text'] = df['clean_text'].apply(lambda x: dataPreprocessing(x))

   

    #punctuation removal

    def remove_punct(text):
        table=str.maketrans('','',string.punctuation)
        return text.translate(table)
    
    df['no_punct'] = df['clean_text'].apply(lambda x: remove_punct(x))

    #stopwords removal

    stop_words = set(stopwords.words('english'))

    def remove_stopwords(text):
        
        word_tokens = word_tokenize(text)
        sent_tokens = sent_tokenize(text)

        filtered_words = [word for word in word_tokens if word.lower() not in stop_words]

        filtered_sent = [word for word in sent_tokens if word.lower() not in stop_words]

        words = ' '.join(filtered_words)

        sents = ' '.join(filtered_sent)

        return words, sents
    
    df['word_tokens'], df['sent_tokens'] = zip(*df['clean_text'].map(remove_stopwords))

    return df

In [12]:
def paragraph_text(df, col = 'full_text'):

    df['sent_text'] = df[col].str.split('.')

    return df


train = paragraph_text(train)

train = train.explode('sent_text')

train.head()

,essay_id,full_text,score,sent_text
0,000d118,Many people have car where they live. The thin...,3,Many people have car where they live
0,000d118,Many people have car where they live. The thin...,3,The thing they don't know is that when you us...
0,000d118,Many people have car where they live. The thin...,3,"Street parkig ,driveways and home garages are..."
0,000d118,Many people have car where they live. The thin...,3,You probaly won't see a car in Vauban's stree...
0,000d118,Many people have car where they live. The thin...,3,"The vauban people completed this in 2006 ,the..."


In [13]:
# time the run time

import time

start = time.time()

# Clean the train and test text data

train = clean_text(train, col_name = 'sent_text')


end = time.time()

print(f"Time taken to clean text: {end - start} seconds")

Starting text cleaning process. 

Time taken to clean text: 99.12127089500427 seconds


In [14]:
train

,essay_id,full_text,score,sent_text,lowered,clean_text,no_punct,word_tokens,sent_tokens
0,000d118,Many people have car where they live. The thin...,3,Many people have car where they live,many people have car where they live,many people have car where they live,many people have car where they live,many people car live,many people have car where they live
0,000d118,Many people have car where they live. The thin...,3,The thing they don't know is that when you us...,the thing they don't know is that when you us...,the thing they do not know is that when you us...,the thing they do not know is that when you us...,thing know use car alot thing happen like get ...,the thing they do not know is that when you us...
0,000d118,Many people have car where they live. The thin...,3,"Street parkig ,driveways and home garages are...","street parkig ,driveways and home garages are...","street parkig, driveways and home garages are ...",street parkig driveways and home garages are f...,"street parkig , driveways home garages forbidd...","street parkig, driveways and home garages are ..."
0,000d118,Many people have car where they live. The thin...,3,You probaly won't see a car in Vauban's stree...,you probaly won't see a car in vauban's stree...,you probaly will not see a car in vauban's str...,you probaly will not see a car in vaubans stre...,probaly see car vauban 's streets completely `...,you probaly will not see a car in vauban's str...
0,000d118,Many people have car where they live. The thin...,3,"The vauban people completed this in 2006 ,the...","the vauban people completed this in 2006 ,the...",the vauban people completed this in two thousa...,the vauban people completed this in two thousa...,"vauban people completed two thousand six , sai...",the vauban people completed this in two thousa...
...,...,...,...,...,...,...,...,...,...
17306,fffed3e,Venus is worthy place to study but dangerous. ...,2,\n\nJust like when they sent a robot to the pl...,\n\njust like when they sent a robot to the pl...,just like when they sent a robot to the planet...,just like when they sent a robot to the planet...,like sent robot planet material tthat withstan...,just like when they sent a robot to the planet...
17306,fffed3e,Venus is worthy place to study but dangerous. ...,2,", But that only lasted two weeks",", but that only lasted two weeks",", but that only lasted two weeks",but that only lasted two weeks,", lasted two weeks",", but that only lasted two weeks"
17306,fffed3e,Venus is worthy place to study but dangerous. ...,2,Now they are try to go for longer the two weeks,now they are try to go for longer the two weeks,now they are try to go for longer the two weeks,now they are try to go for longer the two weeks,try go longer two weeks,now they are try to go for longer the two weeks
17306,fffed3e,Venus is worthy place to study but dangerous. ...,2,\n\nIn Conclusion i know that tcan do it its j...,\n\nin conclusion i know that tcan do it its j...,in conclusion i know that tcan do it its just ...,in conclusion i know that tcan do it its just ...,conclusion know tcan going take time happen,in conclusion i know that tcan do it its just ...


## currently preprocessing removes numbers:

    70 percent of vauban's families do not own cars,and 57 percent sold

    percent of vauban's families do not own cars, and percent sold

In [15]:
print(train['lowered'][0])

0                 many people have car where they live
0     the thing they don't know is that when you us...
0     street parkig ,driveways and home garages are...
0     you probaly won't see a car in vauban's stree...
0     the vauban people completed this in 2006 ,the...
0     the current efforts to drastically reduce gre...
0     i honeslty think that good idea that they did...
0     in the artical david gold berg said that "all...
0     it good that they are doing that if you thik ...
0     in the united states ,the environmental prote...
0     maany experts expect pubic transport serving ...
0     in previous bill,80 percent of appropriations...
0       there many good reason why they should do this
0                                                     
Name: lowered, dtype: object


In [16]:
print(train['clean_text'][0])

0                 many people have car where they live
0    the thing they do not know is that when you us...
0    street parkig, driveways and home garages are ...
0    you probaly will not see a car in vauban's str...
0    the vauban people completed this in two thousa...
0    the current efforts to drastically reduce gree...
0    i honeslty think that good idea that they did ...
0    in the artical david gold berg said that "all ...
0    it good that they are doing that if you thik a...
0    in the united states, the environmental protec...
0    maany experts expect pubic transport serving s...
0    in previous bill, eighty percent of appropriat...
0       there many good reason why they should do this
0                                                     
Name: clean_text, dtype: object


In [17]:
glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name='clean_text')

Loading embeddings from /home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txtLoading embeddings from /home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vecLoading embeddings from /home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt




Reading Embedding File: 999995it [00:28, 35503.17it/s]


Loaded embeddings from /home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec


Reading Embedding File: 1703756it [01:13, 23251.67it/s]


Loaded embeddings from /home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt


...

Coverage checked.
Checking coverage.


Checking Words: 100%|██████████| 92160/92160 [00:00<00:00, 1568778.64it/s]


Coverage checked.
Checking coverage.


Checking Words: 100%|██████████| 92160/92160 [00:00<00:00, 1693161.41it/s]


Coverage checked.
Processed dataset.


In [18]:
misspellings = []
oov = glove + paragram + fastetxt

for word, _ in oov:
    misspellings.append(word)
    misspellings = list(set(misspellings))

print(f"Number of misspelled words: {len(misspellings)}")

# print(f"Misspelled words: {misspellings}")

In [ ]:
from spellchecker import SpellChecker
from tqdm.contrib.concurrent import process_map  # If this import fails, you might need to update tqdm
import multiprocessing

def correct_spellings_batch(misspelled_words_batch):
    """
    Corrects the spelling of words in a batch.

    :param misspelled_words_batch: A batch of misspelled words to be corrected.
    :return: A list of tuples where each tuple contains the original word, the corrected word
             (or None if no correction was found), and a boolean indicating whether the word was corrected.
    """
    spell_checker = SpellChecker()

    results = []
    for word in misspelled_words_batch:
        corrected = spell_checker.correction(word)
        is_corrected = corrected != word and corrected is not None
        result = (word, corrected if is_corrected else None, is_corrected)
        results.append(result)

    return results

def main(misspelled_words):
    num_batches = multiprocessing.cpu_count()

    words_per_batch = len(misspelled_words) // num_batches
    batches = [misspelled_words[i:i + words_per_batch] for i in range(0, len(misspelled_words), words_per_batch)]

    results = process_map(correct_spellings_batch, batches, max_workers=num_batches)

    # Filter to include only corrected words and exclude where corrected is None
    corrected_words = [(original, corrected) for sublist in results for original, corrected, is_corrected in sublist if is_corrected and corrected is not None]
    uncorrected_words = [original for sublist in results for original, corrected, is_corrected in sublist if not is_corrected or corrected is None]

    return corrected_words, uncorrected_words

# Example usage
corrected_words, uncorrected_words = main(misspellings)

print(f"Corrected: {len(corrected_words)} words, Uncorrected: {len(uncorrected_words)} words.")


  0%|          | 0/21 [00:00<?, ?it/s]

Corrected: 53307 words, Uncorrected: 9369 words.


In [ ]:
def apply_corrections_to_text(text, corrections):
    """
    Applies spelling corrections to the text.

    :param text: The original text to be corrected.
    :param corrections: A dictionary of original to corrected word mappings.
    :return: The text with applied spelling corrections.
    """
    words = text.split()  # Tokenize the text into words
    corrected_text = ' '.join([corrections.get(word, word) for word in words])
    return corrected_text

# Assuming df is your DataFrame and 'clean_text' is the column you want to correct

correction_dict = dict(corrected_words)

train['corrected_text'] = train['clean_text'].apply(lambda x: apply_corrections_to_text(x, correction_dict))


In [ ]:
print(f"Corrected words: {len(corrected_words)}")
print(f"Uncorrected words: {len(uncorrected_words)}")

Corrected words: 53307
Uncorrected words: 9369


In [50]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM
import re
import pandas as pd
from multiprocessing import Pool, Manager
import functools
from tqdm import tqdm

def bert_spell_check_text(args):
    text, column_name, misspelled_words, tokenizer, fill_mask, max_length, corrections_dict = args
    corrections = {}
    corrected_text = text
    regex_patterns = {word: re.compile(rf'\b{re.escape(word)}\b') for word in misspelled_words}

    for word, pattern in regex_patterns.items():
        if re.search(pattern, corrected_text):
            sentences = re.split(r'(\.|\?|!)\s+', corrected_text)
            for i, sentence in enumerate(sentences):
                if word in sentence:
                    tokens = tokenizer.tokenize(sentence)
                    if len(tokens) > max_length:
                        continue

                    masked_sentence = re.sub(pattern, tokenizer.mask_token, sentence, count=1)
                    predictions = fill_mask(masked_sentence)
                    if predictions:
                        best_prediction = predictions[0]['sequence']
                        split_prediction = best_prediction.split(tokenizer.mask_token)
                        if len(split_prediction) > 1:
                            corrected_piece = split_prediction[1].strip()
                            if word not in corrections_dict:
                                corrections_dict[word] = set()
                            corrections_dict[word].add((text, corrected_piece))
                            sentences[i] = re.sub(pattern, corrected_piece, sentence, count=1)
            corrected_text = ''.join(sentences)
    return corrected_text

def bert_spell_check_df_parallel(df, column_name, misspelled_words):
    model_name = 'distilbert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    fill_mask = pipeline('fill-mask', model=model, tokenizer=tokenizer)
    max_length = tokenizer.model_max_length

    manager = Manager()
    corrections_dict = manager.dict()

    args = [(row, column_name, misspelled_words, tokenizer, fill_mask, max_length, corrections_dict) for row in df[column_name]]

    with Pool() as pool:
        # Use tqdm with imap_unordered to show progress
        result = list(tqdm(pool.imap(bert_spell_check_text, args), total=len(args)))

    df[f'{column_name}_corrected'] = result
    return df, dict(corrections_dict)

# Example usage
# Assuming `train` is your DataFrame and `clean_text` is the column to be corrected
df, corrections = bert_spell_check_df_parallel(train, 'corrected_text', uncorrected_words)
print("DataFrame with corrections applied.")
print("Corrections:", corrections)


2024-04-11 22:57:37.774371: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-11 22:57:37.804244: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-11 22:57:38.525667: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


NameError: name 'train' is not defined

In [ ]:
print(f"Number of corrections: {len(corrections)}")

NameError: name 'corrections' is not defined

## merge exploded df back together

In [ ]:

result = train.groupby('essay_id').agg({
    'corrected_text_corrected': ' '.join,  
    'corrected_text': ' '.join }).reset_index()


# Merging back other columns if needed, here using 'id' from the original 'train'

result = result.merge(train[['essay_id', 'score']].drop_duplicates('essay_id').drop(columns=['sent_text', 'full_text', 
                                                                                             'lowered', 'no_punct',
                                                                                             'word_tokens', 'sent_tokens']), on='essay_id')


In [ ]:
result.head()

In [ ]:
result = clean_text(result, col_name = 'corrected_text_corrected')

In [17]:
print(result['corrected_text'][0])

Many people have car where they live. The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders. You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that this an example of a growing trend in Europe,The untile states and some where else ar

In [18]:
print(result['corrected_text_corrected'][0])

many people have car where they live .  the thing they do not know is that when you use a car alot of thing can happen like you can get in accident or the smoke that the car has is bad to breath on if someone is walk but in  , germany they dont have that proble because 70 percent of   '  s families do not own cars , and 57 percent sold a car to move there .  street parking  , driveways and home garages are forbidden on the outskirts of  that near the french and swiss borders .  you probaly will not see a car in   '  s streets because they are completely   "  car free  "   but if some that lives in  that owns a car ownership is allowed , but there are only two places that you can park a large garages at the edge of the development , where a car owner buys a space but it not cheap to buy one they sell the space for you car for  $ 40 , 000 along with a home .  the  people completed this in 2006  , they said that this an example of a growing trend in europe , the untile states and some whe

In [ ]:
def custom_train_validation_split(essays, test_size=0.2, random_state=56):
    
    """
    Custom function to perform train-validation split ensuring that
    the same prompt IDs are in both training and validation sets.

    Parameters:
    - summaries: DataFrame containing summaries and associated prompt_ids
    - prompts: DataFrame containing prompts and associated prompt_ids
    - test_size: Proportion of the dataset to be used as the validation set
    - random_state: Random seed for reproducibility

    Returns:
    - train_summaries: Training set containing summaries
    - validation_summaries: Validation set containing summaries
    - train_prompts: Training set containing prompts
    - validation_prompts: Validation set containing prompts
    """
    from sklearn.model_selection import train_test_split
    # Extract unique prompt IDs
    unique_essay_ids = essays['essay_id'].unique()

    # Split the unique prompt IDs into training and validation sets
    train_ids, validation_ids = train_test_split(unique_essay_ids, test_size=test_size, random_state=random_state)

    # Use these IDs to filter the original summaries and prompts DataFrames
    train_essays = essays[essays['essay_id'].isin(train_ids)]
    validation_essays = essays[essays['essay_id'].isin(validation_ids)]

    return train_essays, validation_essays


train, validation = custom_train_validation_split(train, 0.25)

In [19]:
# Save the cleaned text data to a Parquet file

train.to_parquet('clean_train.parquet')
validation.to_parquet('clean_validation.parquet')

# test.to_parquet('cleaned_test.parquet')

In [ ]:
# save